# Brain Tumor Segmentation — Prikaz rezultata istreniranog modela

Ovaj notebook **NE trenira** — samo ucitava vec istreniran model i prikazuje rezultate:
1. Ucitava checkpoint (`best_model.pth`)
2. Ucitava vec izracunate metrike iz CSV-a
3. Prikazuje Dice i HD95 po BraTS regijama (WT, TC, ET)
4. Za vizualizacije pokrece inference samo na par pacijenata

In [ ]:
import sys
from pathlib import Path

ROOT = Path(".").resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("inline")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import ListedColormap
import torch

from brats_config.config import OUTPUT_DIR, PATCH_SIZE
from src.dataset import get_patient_dicts, train_val_split
from src.model import build_model
from src.postprocess import postprocess
from src.transforms import get_val_transforms
from src.evaluate import pred_to_regions, compute_dice, compute_hd95
from monai.data import DataLoader, Dataset
from monai.inferers import sliding_window_inference

CHECKPOINT = ROOT / "outputs" / "best_model.pth"
CSV_PATH   = OUTPUT_DIR / "evaluation_results.csv"
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEG_CMAP      = ListedColormap(["black", "#3399FF", "#FF9800", "#FF3333"])
REGION_COLORS = {"WT": "#4CAF50", "TC": "#FF9800", "ET": "#F44336"}

print(f"Device: {DEVICE}")
print(f"Checkpoint: {CHECKPOINT}  — postoji: {CHECKPOINT.exists()}")
print(f"CSV rezultati: {CSV_PATH}  — postoji: {CSV_PATH.exists()}")

## 1. Ucitaj model

In [ ]:
model = build_model().to(DEVICE)
model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model ucitan: {n_params:,} parametara")
print("Spreman za inference (nema treniranja).")

## 2. Ucitaj metrike iz CSV-a

CSV je generiran pokretanjem `python src/evaluate.py` nakon treniranja.  
Ako CSV ne postoji, pokreni `src/evaluate.py` da ga generiras.

In [ ]:
assert CSV_PATH.exists(), (
    f"CSV ne postoji: {CSV_PATH}\n"
    "Pokreni: python src/evaluate.py --checkpoint outputs/best_model.pth"
)

df = pd.read_csv(CSV_PATH)
print(f"Ucitano {len(df)} pacijenata iz {CSV_PATH.name}")
df.head()

## 3. Sumarni rezultati

In [ ]:
metric_cols = [c for c in df.columns if c != "patient_id"]
df_clean    = df[metric_cols].replace([np.inf, -np.inf], np.nan)

print("=" * 60)
print("REZULTATI EVALUACIJE — best_model.pth")
print("=" * 60)
for region in ["WT", "TC", "ET"]:
    d = df_clean[f"Dice_{region}"].dropna()
    h = df_clean[f"HD95_{region}"].dropna()
    print(f"  {region}  Dice: {d.mean():.3f} ± {d.std():.3f}  "
          f"| HD95: {h.mean():.1f} ± {h.std():.1f}")
print("=" * 60)

display(df_clean.describe().round(3))

## 4. Tablica po pacijentu (bojana)

In [ ]:
dice_cols = ["Dice_WT", "Dice_TC", "Dice_ET"]
hd_cols   = ["HD95_WT",  "HD95_TC",  "HD95_ET"]

def color_dice(val):
    if pd.isna(val):  return "background-color: #f0f0f0"
    if val >= 0.8:    return "background-color: #c8e6c9; font-weight: bold"
    if val >= 0.6:    return "background-color: #fff9c4"
    if val >= 0.4:    return "background-color: #ffe0b2"
    return "background-color: #ffcdd2"

disp = df.copy()
disp[hd_cols] = disp[hd_cols].replace([np.inf, -np.inf], np.nan)

display(
    disp.style
    .applymap(color_dice, subset=dice_cols)
    .format({c: "{:.3f}" for c in dice_cols + hd_cols})
)

## 5. Boxplot Dice i HD95

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, prefix, ylabel, ylim in [
    (axes[0], "Dice", "Dice Score",      (-0.05, 1.05)),
    (axes[1], "HD95", "HD95 (vokseli)",  None),
]:
    cols = [f"{prefix}_{r}" for r in ["WT", "TC", "ET"]]
    data = [df[c].replace([np.inf, -np.inf], np.nan).dropna().values for c in cols]

    bp = ax.boxplot(data, labels=["WT", "TC", "ET"],
                    patch_artist=True, widths=0.5)
    for patch, color in zip(bp["boxes"], REGION_COLORS.values()):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    for med in bp["medians"]:
        med.set_color("black")
        med.set_linewidth(2)

    for j, d in enumerate(data):
        ax.scatter(np.random.normal(j+1, 0.07, len(d)), d,
                   alpha=0.4, s=20, color="dimgray", zorder=3)

    ax.set_title(f"{prefix} po BraTS regijama", fontsize=12)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.3)
    if ylim:
        ax.set_ylim(ylim)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eval_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Histogram Dice scoreova

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, region, color in zip(axes, ["WT", "TC", "ET"], REGION_COLORS.values()):
    vals = df[f"Dice_{region}"].replace([np.inf, -np.inf], np.nan).dropna()
    ax.hist(vals, bins=20, color=color, alpha=0.85, edgecolor="white")
    ax.axvline(vals.mean(),   color="navy",    lw=2, ls="--", label=f"mean={vals.mean():.3f}")
    ax.axvline(vals.median(), color="darkred", lw=2, ls=":",  label=f"median={vals.median():.3f}")
    ax.set_title(f"Dice {region}", fontsize=12)
    ax.set_xlabel("Dice Score")
    ax.set_ylabel("Broj pacijenata")
    ax.set_xlim(0, 1)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle("Distribucija Dice Scoreova", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "eval_histograms.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Vizualizacija: Ground Truth vs Predikcija

Inference samo na **N_VIZ pacijenata** (ne na svim) — ovo je brzo.

In [ ]:
# Koliko pacijenata vizualizirati (mali broj — samo za slike)
N_VIZ = 5

all_dicts = get_patient_dicts()
_, val_dicts = train_val_split(all_dicts)
viz_dicts = val_dicts[:N_VIZ]

print(f"Inference na {N_VIZ} pacijenata za vizualizaciju...")

viz_data = []  # lista (flair, gt, pred, patient_id)

for patient_dict in viz_dicts:
    pid = Path(patient_dict["flair"]).parts[-2]

    ds     = Dataset(data=[patient_dict], transform=get_val_transforms())
    loader = DataLoader(ds, batch_size=1, num_workers=0)
    batch  = next(iter(loader))

    inputs = batch["image"].to(DEVICE)
    labels = batch["seg"].squeeze().cpu().numpy().astype(np.int32)

    with torch.no_grad():
        outputs = sliding_window_inference(
            inputs=inputs, roi_size=PATCH_SIZE,
            sw_batch_size=1, predictor=model, overlap=0.25,
        )

    pred = outputs.argmax(dim=1).squeeze().cpu().numpy().astype(np.int32)
    pred = postprocess(pred, min_voxels=50)

    flair = inputs[0, 0].cpu().numpy()
    viz_data.append((flair, labels, pred, pid))
    print(f"  OK: {pid}")

print("Gotovo.")

In [ ]:
def find_tumor_slices(seg, n=3):
    counts = (seg > 0).sum(axis=(0, 1))
    if counts.sum() == 0:
        z = seg.shape[2] // 2
        return [z]
    z_idx = np.where(counts > counts.max() * 0.05)[0]
    pcts  = [25, 50, 75] if n == 3 else [50]
    return [int(np.percentile(z_idx, p)) for p in pcts]


def plot_patient(flair, gt, pred, pid, dice_row):
    zs = find_tumor_slices(gt, n=3)
    fig = plt.figure(figsize=(5 * len(zs), 11))
    gs  = gridspec.GridSpec(3, len(zs), hspace=0.06, wspace=0.04)

    for col, z in enumerate(zs):
        for row, (data, cmap, vmin, vmax) in enumerate([
            (flair[:, :, z].T, "gray",   None, None),
            (gt[:, :, z].T,    SEG_CMAP, 0,    3),
            (pred[:, :, z].T,  SEG_CMAP, 0,    3),
        ]):
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax,
                      origin="lower", interpolation="nearest")
            ax.axis("off")
            if col == 0:
                label = ["FLAIR", "Ground Truth", "Predikcija"][row]
                ax.set_ylabel(label, fontsize=11)
                ax.yaxis.set_label_coords(-0.08, 0.5)
            if row == 0:
                ax.set_title(f"z = {z}", fontsize=10)

    patches = [
        mpatches.Patch(color=c, label=n)
        for c, n in zip(["#3399FF", "#FF9800", "#FF3333"],
                        ["NCR/Nekroza", "Edem", "ET (Enhancing)"])
    ]
    fig.legend(handles=patches, loc="lower center", ncol=3,
               fontsize=10, bbox_to_anchor=(0.5, -0.02))

    hd = dice_row
    fig.suptitle(
        f"{pid}\n"
        f"Dice  WT={hd['Dice_WT']:.3f}  TC={hd['Dice_TC']:.3f}  ET={hd['Dice_ET']:.3f}\n"
        f"HD95  WT={hd['HD95_WT']:.1f}  TC={hd['HD95_TC']:.1f}  ET={hd['HD95_ET']:.1f}",
        fontsize=11, y=1.01
    )
    plt.savefig(OUTPUT_DIR / f"viz_{pid}.png", dpi=120, bbox_inches="tight")
    plt.show()
    plt.close()


for flair, gt, pred, pid in viz_data:
    row_match = df[df["patient_id"] == pid]
    if row_match.empty:
        # Pacijent nije u CSV-u — izracunaj metrike na licu mjesta
        row = {"patient_id": pid}
        pr = pred_to_regions(torch.from_numpy(pred))
        gr = pred_to_regions(torch.from_numpy(gt))
        for reg in ["WT", "TC", "ET"]:
            p = pr[reg].numpy().astype(bool)
            g = gr[reg].numpy().astype(bool)
            row[f"Dice_{reg}"] = compute_dice(p, g)
            row[f"HD95_{reg}"] = compute_hd95(p, g)
        dice_row = pd.Series(row)
    else:
        dice_row = row_match.iloc[0]

    plot_patient(flair, gt, pred, pid, dice_row)

## 8. BraTS regije (WT / TC / ET) — GT vs Predikcija

In [ ]:
def plot_regions(flair, gt, pred, pid):
    z = find_tumor_slices(gt, n=1)[0]
    gt_reg   = pred_to_regions(torch.from_numpy(gt))
    pred_reg = pred_to_regions(torch.from_numpy(pred))

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    fig.suptitle(f"{pid}  |  z = {z}  — GT vs Predikcija po regijama", fontsize=12)

    flair_sl = flair[:, :, z].T

    for row_idx, (regions, row_label) in enumerate([
        (gt_reg,   "Ground Truth"),
        (pred_reg, "Predikcija"),
    ]):
        src_mask = gt if row_idx == 0 else pred

        axes[row_idx, 0].imshow(flair_sl, cmap="gray", origin="lower")
        axes[row_idx, 0].set_title("FLAIR")
        axes[row_idx, 0].set_ylabel(row_label, fontsize=11)

        axes[row_idx, 1].imshow(src_mask[:, :, z].T, cmap=SEG_CMAP,
                                vmin=0, vmax=3, origin="lower")
        axes[row_idx, 1].set_title("Sve klase")

        for col, (region, color) in enumerate(REGION_COLORS.items(), start=2):
            mask = regions[region].numpy()[:, :, z].T
            axes[row_idx, col].imshow(flair_sl, cmap="gray", origin="lower", alpha=0.45)
            axes[row_idx, col].imshow(
                np.ma.masked_where(mask == 0, mask),
                cmap=ListedColormap([color]), origin="lower",
                alpha=0.85, vmin=0, vmax=1
            )
            axes[row_idx, col].set_title(f"{region}")

    for ax in axes.flat:
        ax.axis("off")

    legend = [
        mpatches.Patch(color=c, label=r)
        for r, c in REGION_COLORS.items()
    ]
    fig.legend(handles=legend, loc="lower center", ncol=3,
               fontsize=10, bbox_to_anchor=(0.5, -0.02))
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"regions_{pid}.png", dpi=130, bbox_inches="tight")
    plt.show()
    plt.close()


for flair, gt, pred, pid in viz_data:
    plot_regions(flair, gt, pred, pid)

## 9. Najbolji i najlosiji pacijenti

In [ ]:
df_s = df.sort_values("Dice_WT", ascending=False)

print("Top 5 pacijenata (Dice_WT):")
display(df_s[["patient_id", "Dice_WT", "Dice_TC", "Dice_ET"]].head(5).round(3))

print("\nNajgori 5 pacijenata (Dice_WT):")
display(df_s[["patient_id", "Dice_WT", "Dice_TC", "Dice_ET"]].tail(5).round(3))